In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from rdkit.Chem.MolStandardize import rdMolStandardize

In [ ]:
data = pd.read_csv(
    "/Users/ernestofadel/Desktop/Total_data.csv",       #Local file with all data from the SQL database. Is too big to put it in the gitlab repository
    header=0,
    names=[
        "Assay_id",
        "Description",
        "Document_id",
        "Document_year",
        "Authors",
        "Confidence_score",
        "Target_id",
        "Pref_name",
        "Protein_sequence",
        "Uniprot_id",
        "Variant_id",
        "Molecule_id",
        "Canonical_smile",
        "Standard_type",
        "Standard_value",
        "Standard_units",
        "pchembl_value",
        "assay_type",
        "assay_organism",
        "assay_category",
        "assay_tax_id",
        "assay_strain",
        "assay_tissue",
        "assay_cell_type",
        "assay_subcellular_fraction",
        "bao_format",
    ],
    low_memory=False,
)

data

,Assay_id,Description,Document_id,Document_year,Authors,Confidence_score,Target_id,Pref_name,Protein_sequence,Uniprot_id,...,pchembl_value,assay_type,assay_organism,assay_category,assay_tax_id,assay_strain,assay_tissue,assay_cell_type,assay_subcellular_fraction,bao_format
0,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,"Courtney SM, Hay PA, Buck RT, Colville CS, Por...",8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,...,5.60,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
1,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,"Courtney SM, Hay PA, Buck RT, Colville CS, Por...",8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,...,5.05,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
2,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,"Koltun DO, Marquart TA, Shenk KD, Elzein E, Li...",8,CHEMBL4632,Palmitoyl-CoA oxidase,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,...,5.40,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
3,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,"Koltun DO, Marquart TA, Shenk KD, Elzein E, Li...",8,CHEMBL4632,Palmitoyl-CoA oxidase,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,...,4.77,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
4,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,"Koltun DO, Marquart TA, Shenk KD, Elzein E, Li...",8,CHEMBL4632,Palmitoyl-CoA oxidase,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,...,6.75,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
932476,CHEMBL5218239,Inhibition of N-terminal His-tagged recombinan...,CHEMBL5215002,2022.0,"Zhang Z, Shang ZP, Jiang Y, Qu ZX, Yang RY, Zh...",9,CHEMBL335,Protein-tyrosine phosphatase 1B,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,P18031,...,4.86,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
932477,CHEMBL5218239,Inhibition of N-terminal His-tagged recombinan...,CHEMBL5215002,2022.0,"Zhang Z, Shang ZP, Jiang Y, Qu ZX, Yang RY, Zh...",9,CHEMBL335,Protein-tyrosine phosphatase 1B,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,P18031,...,4.97,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
932478,CHEMBL5218239,Inhibition of N-terminal His-tagged recombinan...,CHEMBL5215002,2022.0,"Zhang Z, Shang ZP, Jiang Y, Qu ZX, Yang RY, Zh...",9,CHEMBL335,Protein-tyrosine phosphatase 1B,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,P18031,...,5.53,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
932479,CHEMBL5218240,Inhibition of recombinant human TCPTP (1 to 31...,CHEMBL5215002,2022.0,"Zhang Z, Shang ZP, Jiang Y, Qu ZX, Yang RY, Zh...",9,CHEMBL3807,T-cell protein-tyrosine phosphatase,MPTTIEREFEELDTQRRWQPLYLEIRNESHDYPHRVAKFPENRNRN...,P17706,...,4.11,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000219


In [3]:
# Filter by assay size


def filter_by_size(data, minAssaySize, maxAssaySize):

    count = data.groupby("Assay_id")["Molecule_id"].nunique()

    filtered = count[(count >= minAssaySize) & (count <= maxAssaySize)]

    df = data[data["Assay_id"].isin(filtered.index)]

    return df


# Removing non unique protein sequence


def remove_non_unique_sequences(data):

    target_info = data[
        ["Target_id", "Protein_sequence", "Uniprot_id"]
    ].drop_duplicates()

    sequence_count = (
        target_info.groupby("Protein_sequence")["Target_id"]
        .agg([("Target_count", "nunique"), ("targets", lambda x: list(x.unique()))])
        .reset_index()
    )

    non_unique_sequences = sequence_count[sequence_count["Target_count"] > 1]

    targets_to_remove = set()
    for _, row in non_unique_sequences.iterrows():
        targets_to_remove.update(row["targets"])

    df_filtered = data[~data["Target_id"].isin(targets_to_remove)]

    return df_filtered


# Removing mutated protein


def remove_mutant(data):

    mask = pd.isna(data["Variant_id"])

    data = data[mask]

    exclude_keywords = ["mutant", "mutation", "variant"]

    pattern = "|".join(exclude_keywords)

    df = data[~data["Description"].str.contains(pattern, case=False, na=False)]

    return df


# Removing data with confidence score lower than 9


def remove_low_confidence(data):

    mask = data["Confidence_score"] == 9

    df = data[mask]

    return df


# Removing data with missing documentation (no document year)


def remove_missing_doc_year(data):

    mask = pd.isna(data["Document_year"])

    df = data[~mask]

    return df


# Calculating missing pchembl_value


def fill_pchembl_values(data):

    df = data

    mask = df["pchembl_value"].isna()

    df.loc[mask, "pchembl_value"] = 9 - np.log10(df.loc[mask, "Standard_value"])

    return df


# Creating standard smiles from smiles


def stdandardize_smiles(df):

    df["Canonical_smile"] = df["Canonical_smile"].apply(
        rdMolStandardize.StandardizeSmiles
    )

    return df

In [ ]:
# Extracting Landrum dataset from raw data

Data = remove_low_confidence(data)
Data = remove_mutant(Data)
Data = remove_missing_doc_year(Data)
Data = filter_by_size(Data, 20, 100)
Landrum = remove_non_unique_sequences(Data)

# Final remove NaN value still present

Landrum = Landrum[Landrum["Canonical_smile"].notna()]
Landrum = Landrum[Landrum["Standard_value"].notna()]
Landrum = stdandardize_smiles(Landrum)

# Fill missing pIC50 value and removing all column not necessary for ML

Landrum = fill_pchembl_values(Landrum)
Landrum = Landrum[
    [
        "Assay_id",
        "Target_id",
        "Protein_sequence",
        "Molecule_id",
        "Canonical_smile",
        "Standard_value",
        "pchembl_value",
    ]
]
Landrum = Landrum.rename(columns={"Standard_value": "IC50", "pchembl_value": "pIC50"})

Landrum

,Assay_id,Target_id,Protein_sequence,Molecule_id,Canonical_smile,IC50,pIC50
20,CHEMBL641889,CHEMBL4464,MSLRNRLSKSGENPEQDEAQKNFMDTYRNGHITMKQLIAKKRLLAA...,CHEMBL50381,COc1ccc(-c2nc(SCCCCCN(CCCn3ccnc3)C(=O)NC(C)C)[...,5500.00,5.26
22,CHEMBL641889,CHEMBL4464,MSLRNRLSKSGENPEQDEAQKNFMDTYRNGHITMKQLIAKKRLLAA...,CHEMBL431940,CN(C)c1ccc(-c2nc(SCCCCCN(CCCCCSc3nc(-c4ccccc4)...,450.00,6.35
68,CHEMBL645934,CHEMBL285,MVGEETSLRNRLSRSAENPEQDEAQKNLLDTHRNGHITMKQLIAKK...,CHEMBL304930,Cc1cc(C)c2ncc(NC(=O)Nc3c(F)cc(F)cc3F)c(-c3cccc...,15.00,7.82
74,CHEMBL769366,CHEMBL243,PQVTLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKM...,CHEMBL108102,COC(=O)N[C@H](C(=O)N[C@@H](Cc1ccccc1)C(O)CN(Cc...,42.00,7.38
88,CHEMBL701719,CHEMBL3471,FLDGIDKAQDEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLK...,CHEMBL324842,O=C(/C=C/c1ccc(O)c(O)c1)O[C@H](Cc1ccc(O)c(O)c1...,9000.00,5.05
...,...,...,...,...,...,...,...
932462,CHEMBL5218150,CHEMBL1649052,MKVSAALLCLLLIAATFIPQGLAQPDAINAPVTCCYNFTNRKISVQ...,CHEMBL5220128,C[C@H]1C(=O)N(C)C=C(c2cn[nH]c2)c2cc([C@@H](O)C...,31.62,7.50
932463,CHEMBL5218150,CHEMBL1649052,MKVSAALLCLLLIAATFIPQGLAQPDAINAPVTCCYNFTNRKISVQ...,CHEMBL5218909,C[C@H]1C(=O)N(C)C=C(c2cnn(C)c2)c2cc([C@H](O)CO...,39.81,7.40
932464,CHEMBL5218150,CHEMBL1649052,MKVSAALLCLLLIAATFIPQGLAQPDAINAPVTCCYNFTNRKISVQ...,CHEMBL5219602,C[C@H]1C(=O)N(C)C=C(c2cnn(C)c2)c2cc([C@@H](O)C...,39.81,7.40
932465,CHEMBL5218150,CHEMBL1649052,MKVSAALLCLLLIAATFIPQGLAQPDAINAPVTCCYNFTNRKISVQ...,CHEMBL5219538,Cc1nc(C2=CN(C)C(=O)[C@H](C)c3ccc(OCCO)cc32)c[nH]1,39.81,7.40


In [ ]:
# Extracting Omnivore dataset from raw data

Data = remove_mutant(data)
Omnivore = remove_non_unique_sequences(Data)

# Final remove NaN value still present

Omnivore = Omnivore[Omnivore["Canonical_smile"].notna()]
Omnivore = Omnivore[Omnivore["Standard_value"].notna()]
Omnivore = stdandardize_smiles(Omnivore)

# Fill missing pIC50 value and removing all column not necessary for ML

Omnivore = fill_pchembl_values(Omnivore)
Omnivore = Omnivore[
    [
        "Assay_id",
        "Target_id",
        "Protein_sequence",
        "Molecule_id",
        "Canonical_smile",
        "Standard_value",
        "pchembl_value",
    ]
]
Omnivore = Omnivore.rename(columns={"Standard_value": "IC50", "pchembl_value": "pIC50"})

Omnivore

,Assay_id,Target_id,Protein_sequence,Molecule_id,Canonical_smile,IC50,pIC50
0,CHEMBL872937,CHEMBL3921,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,CHEMBL324340,Cc1ccc2oc(-c3cccc(N4C(=O)c5ccc(C(=O)O)cc5C4=O)...,2500.0,5.60
1,CHEMBL872937,CHEMBL3921,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,CHEMBL109600,COc1ccccc1-c1ccc2oc(-c3ccc(OC)c(N4C(=O)c5ccc(C...,9000.0,5.05
2,CHEMBL760688,CHEMBL4632,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,CHEMBL357278,Cc1nc2cc(OC[C@H](O)CN3CCN(CC(=O)Nc4ccc(Cl)c(C(...,4000.0,5.40
3,CHEMBL760688,CHEMBL4632,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,CHEMBL357119,Cc1nc2cc(OC[C@H](O)CN3CCN(CC(=O)NCCc4ccccc4)CC...,17000.0,4.77
4,CHEMBL760688,CHEMBL4632,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,CHEMBL152968,Cc1nc2cc(OC[C@H](O)CN3CCN(CC(=O)Nc4cccc(-c5ccc...,180.0,6.75
...,...,...,...,...,...,...,...
932474,CHEMBL5218239,CHEMBL335,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,CHEMBL5219224,COc1ccc2c(c1)C(=O)c1cc(O[C@@H]3O[C@H](CO)[C@@H...,1560.0,5.81
932475,CHEMBL5218239,CHEMBL335,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,CHEMBL5219028,COc1ccc2c(c1O)C(=O)c1cc(O[C@@H]3O[C@H](CO)[C@@...,3050.0,5.52
932476,CHEMBL5218239,CHEMBL335,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,CHEMBL5220958,Cc1c(O[C@@H]2O[C@H](CO[C@@H]3OC[C@@H](O)[C@H](...,13740.0,4.86
932477,CHEMBL5218239,CHEMBL335,MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRY...,CHEMBL5220938,O=C1c2ccccc2C(=O)c2c1cc(O[C@@H]1O[C@H](CO)[C@@...,10810.0,4.97


In [10]:
# This curation process closely follows the one presented at https://github.com/openkinome/kinodata/blob/master/kinase-bioactivities-in-chembl/kinase-bioactivities-in-chembl.ipynb

kinases = pd.read_csv("Csv_files/human_kinases_and_chembl_targets.chembl_33.csv")

# Removing all assay_type that are not B


def removing_non_binding_type(data):

    df = data[(data["assay_type"] == "B")]

    return df


def filtering_kinase_id(data, df2=kinases):

    df = data[data["Target_id"].isin(df2["chembl_targets"])]

    return df


def remove_dummy_id(data):

    df = data[data["Target_id"] != "CHEMBL612545"]

    return df


def remove_extreme_value(data):

    df = data[data["pchembl_value"].between(1, 15)]

    return df


def keep_only_max_value(data):

    df = data.sort_values("pchembl_value", ascending=False).drop_duplicates(
        ["Target_id", "Molecule_id", "Document_id"]
    )

    return df


def removing_duplicates(data):

    df = data.drop_duplicates(["Target_id", "Molecule_id", "pchembl_value"])
    df = (
        df.assign(
            activities_standard_value_rounded=lambda x: x["pchembl_value"].round(2)
        )
        .drop_duplicates(
            ["Target_id", "Molecule_id", "activities_standard_value_rounded"]
        )
        .drop(columns=["activities_standard_value_rounded"])
    )

    return df


def shared_authors(group):
    "Return True if authors are not shared and we should keep this group"
    if group.shape[0] == 1:
        return [True]
    authors_per_entry = [
        set(entry.split(", ")) if pd.notna(entry) else set() for entry in group.values
    ]
    return [
        any(a.intersection(b) for b in authors_per_entry if a != b)
        for a in authors_per_entry
    ]


def remove_same_authors(data):

    # df = data.dropna(subset=["Authors"])

    mask = data.groupby(["Target_id", "Molecule_id"])["Authors"].transform(
        shared_authors
    )

    df = data[mask]

    return df

In [ ]:
# Adding curation step to obtain kinodata dataset from raw data

Data = filtering_kinase_id(data)
Data = removing_non_binding_type(Data)
Data = remove_dummy_id(Data)
Data = remove_extreme_value(Data)
Data = keep_only_max_value(Data)
Data = removing_duplicates(Data)
Kinodata = remove_same_authors(Data)

# Final remove NaN value still present

Kinodata = Kinodata[Kinodata["Canonical_smile"].notna()]
Kinodata = Kinodata[Kinodata["Standard_value"].notna()]
Kinodata = stdandardize_smiles(Kinodata)

# Fill missing pIC50 value and removing all column not necessary for ML

Kinodata = fill_pchembl_values(Kinodata)
Kinodata = Kinodata[
    [
        "Assay_id",
        "Target_id",
        "Protein_sequence",
        "Molecule_id",
        "Canonical_smile",
        "Standard_value",
        "pchembl_value",
    ]
]
Kinodata = Kinodata.rename(columns={"Standard_value": "IC50", "pchembl_value": "pIC50"})

Kinodata

,Assay_id,Target_id,Protein_sequence,Molecule_id,Canonical_smile,IC50,pIC50
526971,CHEMBL3705754,CHEMBL2147,MLLSKINSLAHLRAAPCNDLHATKLAPGKEKEPLESQYQVGPLLGS...,CHEMBL3691967,C[C@H]1C[C@@H](c2ccncc2NC(=O)c2ccc(F)c(-c3c(F)...,0.01,11.00
163910,CHEMBL913225,CHEMBL258,MGCGCSSHPEDDWMENIDVCENCHYPIVPLDGKGTLLIRNGSEVRD...,CHEMBL215969,CNc1ncc2cc(-c3cc(C(=O)Nc4cccc(C(F)(F)F)c4C)ccc...,0.01,11.00
666603,CHEMBL3887900,CHEMBL4040,MAAAAAAGAGPEMVRGQVFDVGPRYTNLSYIGEGAYGMVCSAYDNV...,CHEMBL4108345,CSc1cccc([C@@H](CO)NC(=O)c2ccc(-c3nc(C4CCOCC4)...,0.01,11.00
532193,CHEMBL3706327,CHEMBL4722,MDRSKENCISGPVKATAPVGGPKRVLVTQQFPCQNPLPVNSGQAQR...,CHEMBL3685304,C[C@@H](Nc1ncnc2c(C(N)=O)cccc12)c1cccc(NC(=O)c...,0.01,11.00
683529,CHEMBL3888428,CHEMBL2835,MQYLNIKEDCNAMAFCAKMRSSKKTEVNLEAPEPGVEVIFYLSDRE...,CHEMBL3896068,N#CCC1(n2cc(C(N)=O)c(Nc3ccc(-c4cn[nH]c4)nc3)n2...,0.01,11.00
...,...,...,...,...,...,...,...
777051,CHEMBL4316274,CHEMBL1163125,MSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPP...,CHEMBL4466089,CN1CC(CO)CC1=O,4110000.00,2.39
40721,CHEMBL823240,CHEMBL267,MGSNKSKPKDASQRRRSLEPAENVHGAGGGAFPASQTPSKPASADG...,CHEMBL284362,CCc1ccc(NC(=O)P(=O)(O)O)cc1,4400000.00,2.36
777053,CHEMBL4316274,CHEMBL1163125,MSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPP...,CHEMBL4570065,CN1CC(CCO)CC1=O,4810000.00,2.32
75752,CHEMBL809533,CHEMBL267,MGSNKSKPKDASQRRRSLEPAENVHGAGGGAFPASQTPSKPASADG...,CHEMBL309800,Cc1cc(C=O)cc(C)c1OP(=O)(O)O,5000000.00,2.30


In [ ]:
# Converting all curated dataset in csv files

Landrum.to_csv("Csv_files/landrum_dataset.csv")
Omnivore.to_csv("Csv_files/omnivore_dataset.csv")
Kinodata.to_csv("Csv_files/kinodata_dataset.csv")